In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import plotly.express as px

In [2]:
# To make plotly fig show in notebook
import plotly.io as pio
pio.renderers.default = "notebook"

# MAIN

In [49]:
# Read filtered TSV (only non fully '0' rows)
X = pd.read_csv("CUSTOM_hg38_episign/meth_matrix.tsv", sep="\t", index_col=0)

# Compute epiSize (= size of epiSign):
epiSize = {}
for epiSign in [x for x in X.columns if x not in ('coord')]:
    epiSize[epiSign] = sum(X[epiSign] > 99)  # Catch 100% methyl

## Pre-processing

In [50]:
# Transpose
X_t = X.T  # Required
print(X_t.index)

to_PCA = X_t
if 'epiSize' in X_t.columns:
    to_PCA = X_t.drop('epiSize', axis=1)

# Remove 2nd row = size of epiSignormalize) then normalize
X_scaled = StandardScaler().fit_transform(to_PCA)

Index(['ADCADN', 'ATRX', 'AUTS18', 'BAFopathy', 'BFLS', 'CHARGE', 'CdLS',
       'Down', 'Dup7', 'EEOC', 'FLHS', 'GTPTS', 'HMA', 'HVDAS_C', 'HVDAS_T',
       'ICF1', 'ICF2_3_4', 'KDVS', 'Kabuki', 'Kleefstra', 'MRD51', 'MRX93',
       'MRX97', 'MRXCJS', 'MRXSN', 'MRXSSR', 'RMNS', 'RSTS', 'SBBYSS',
       'SETD1B', 'Sotos', 'TBRS', 'WDSTS', 'Williams', 'HG002_combined',
       'barcode04_combined'],
      dtype='object')


## PCA

In [51]:
# Run PCA:
NB_COMPON = 3
pca = PCA(n_components=NB_COMPON)
pcs = pca.fit_transform(X_scaled)

# Make a dict with '% variance explained' for each component:
dict_compon = {'compon'+str(i) : str(round(pca.explained_variance_ratio_[i]*100,4)) for i in range(NB_COMPON)}

In [52]:
# Top N features of each componennt
compon_0_top = np.abs(pca.components_[0]).argsort()[::-1][:5]
print("Component 0:", list(X.index[compon_0_top]))

compon_1_top = np.abs(pca.components_[1]).argsort()[::-1][:5]
print("Component 1:", list(X.index[compon_1_top]))

Component 0: ['14:101635322-101635323', '5:13810085-13810086', '11:121157882-121157883', '10:2936245-2936246', '1:1067995-1067996']
Component 1: ['13:111658372-111658373', '1:160342391-160342392', '7:100589775-100589776', '22:20430254-20430255', '9:19378680-19378681']


In [53]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
pcs_df = pd.DataFrame(
    pcs,
    index=X.columns,
    columns=dict_compon.keys()
)
print(pcs_df.loc[['Kabuki.bed', 'barcode04_combined', 'HG002_combined']])

KeyError: "['Kabuki.bed'] not in index"

In [ ]:
# Plot PCA
color_selected = [ x in ['RMNS.bed','Kleefstra.bed','Kabuki.bed','barcode04_combined','HG002_combined'] for x in pcs_df.index ]

x_compon = 'compon0'
y_compon = 'compon1'

fig = px.scatter(
        pcs_df,
        x=x_compon,
        y=y_compon,
        hover_data=[pcs_df.index],
        color=color_selected,
        labels={x_compon:':'.join([x_compon,dict_compon[x_compon]]), y_compon:':'.join([y_compon,dict_compon[y_compon]])}
)
fig.show()

## t-SNE

In [ ]:
# Run t-SNE:
# MEMOs:
# - In Joris' paper they use 'preplex=2'
# - t-SNE is stochastic -> re-run multiple times ?
#
tsne = TSNE(n_components=2, perplexity=2).fit_transform(X_scaled)

In [ ]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
tsne_df = pd.DataFrame(
    tsne,
    index=X.columns,
    columns=('compon0', 'compon1')
)
subset_tsne = tsne_df.loc[['Kabuki.bed', 'barcode04_combined', 'HG002_combined']]
print(subset_tsne)

In [ ]:
# Plot t-SNE
color_selected = [x in ['RMNS.bed','Kleefstra.bed', 'Kabuki.bed', 'barcode04_combined', 'HG002_combined'] for x in tsne_df.index]
fig = px.scatter(
        tsne_df,
        x='compon0',
        y='compon1',
        hover_data=[tsne_df.index],
        color=color_selected
)
fig.show()